## Validation of Feature Importance with Chi-Square test and Cramér's V.
- I used Chi-Square test to check the independace of features with independance.
    - It gives chi-square value which tells the association strength of feature with the churn.
    - It also gives p value which tells what is the probability of this data as randomness.
- I also used Cramér's V which gives the association stength of feature with churn
    - The Chi-Square value is dependant on the sample size, So we cannot use it for the comparison of association strength inbetween the feature
    - Cramér's V computes the association strength irrespective of sample size

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency


In [ ]:
df = pd.read_csv("../Data/Cleaned/customer_churn_dataset.csv")
df.head()

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,1.0,22.0,Female,25.0,14.0,4.0,27.0,Basic,Monthly,598.0,9.0,1.0
1,2.0,30.0,Female,39.0,14.0,5.0,18.0,Standard,Annual,932.0,17.0,1.0
2,2.0,41.0,Female,28.0,28.0,7.0,13.0,Standard,Monthly,584.0,20.0,0.0
3,3.0,47.0,Male,27.0,10.0,2.0,29.0,Premium,Annual,757.0,21.0,0.0
4,3.0,65.0,Female,49.0,1.0,10.0,8.0,Basic,Monthly,557.0,6.0,1.0


### Chi-Square test

In [ ]:
import numpy as np

df["Issue_Level"] = np.where(
    df["Support Calls"] <= 2,
    "Low Issues",
    np.where(df["Support Calls"] <= 4,
             "Medium Issues",
             "High Issues")
)

df["Delay_Level"] = np.where(
    df["Payment Delay"] <= 15,
    "Low Delay",
    np.where(df["Payment Delay"] <= 20,
             "Medium Delay",
             "High Delay")
)

df["Spend_Level"] = np.where(
    df["Total Spend"] <= 508,
    "Low Spend",
    "High Spend"
)

In [ ]:
from scipy.stats import chi2_contingency

features = [
    "Issue_Level",
    "Delay_Level",
    "Spend_Level",
    "Contract Length"
]

results = []

for feature in features:

    table = pd.crosstab(
        df[feature],
        df["Churn"]
    )

    chi2, p, dof, expected = chi2_contingency(table)

    results.append([
        feature,
        chi2,
        p
    ])

results_df = pd.DataFrame(
    results,
    columns=[
        "Feature",
        "Chi_Square",
        "P_Value"
    ]
)

results_df

,Feature,Chi_Square,P_Value
0,Issue_Level,152535.692764,0.0
1,Delay_Level,87480.684564,0.0
2,Spend_Level,94491.019220,0.0
3,Contract Length,67861.646650,0.0


#### Observation and Conclusion
1. The p value for all 4 most important feature is 0 which rejects the independance of these feature with customers churn.

### Cramér's V.

In [ ]:

def cramers_v(table):

    chi2 = chi2_contingency(table)[0]

    n = table.sum().sum()

    r, k = table.shape

    return np.sqrt(
        chi2 / (n * min(r - 1, k - 1))
    )

In [ ]:
features = [
    "Issue_Level",
    "Delay_Level",
    "Spend_Level",
    "Contract Length"
]

results = []

for feature in features:

    table = pd.crosstab(
        df[feature],
        df["Churn"]
    )

    chi2, p, dof, expected = chi2_contingency(table)

    cv = cramers_v(table)

    results.append([
        feature,
        chi2,
        p,
        cv
    ])

results_df = pd.DataFrame(
    results,
    columns=[
        "Feature",
        "Chi_Square",
        "P_Value",
        "Cramers_V"
    ]
)

results_df.sort_values(
    by="Cramers_V",
    ascending=False
)

,Feature,Chi_Square,P_Value,Cramers_V
0,Issue_Level,152535.692764,0.0,0.549479
2,Spend_Level,94491.019220,0.0,0.432475
1,Delay_Level,87480.684564,0.0,0.416123
3,Contract Length,67861.646650,0.0,0.366503


#### Observation and Conclusion

- The Chi-Square test of independance can check the independance of features with customer churn but it cannot tell how strong is association with churn due to it's bias with sample size
- Cramér's V removes the effect of sample size as we got Cramér's V between 0.36 to 0.54 which suggests strong association with customer churn

## Logistic Regression for Feature Importance,Segment Analysis and ranking 

In [ ]:

from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("../Data/Cleaned/customer_churn_dataset.csv")
df.head(5)

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,1.0,22.0,Female,25.0,14.0,4.0,27.0,Basic,Monthly,598.0,9.0,1.0
1,2.0,30.0,Female,39.0,14.0,5.0,18.0,Standard,Annual,932.0,17.0,1.0
2,2.0,41.0,Female,28.0,28.0,7.0,13.0,Standard,Monthly,584.0,20.0,0.0
3,3.0,47.0,Male,27.0,10.0,2.0,29.0,Premium,Annual,757.0,21.0,0.0
4,3.0,65.0,Female,49.0,1.0,10.0,8.0,Basic,Monthly,557.0,6.0,1.0


In [ ]:

df["Issue_Level"] = np.where(
    df["Support Calls"] <= 2,
    "Low Issues",
    np.where(df["Support Calls"] <= 4,
             "Medium Issues",
             "High Issues")
)

df["Delay_Level"] = np.where(
    df["Payment Delay"] <= 15,
    "Low Delay",
    np.where(df["Payment Delay"] <= 20,
             "Medium Delay",
             "High Delay")
)

df["Spend_Level"] = np.where(
    df["Total Spend"] <= 508,
    "Low Spend",
    "High Spend"
)

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
len(df)

505206

In [ ]:
df["CustomerID"].nunique()

442211

In [ ]:
df["CustomerID"].duplicated().sum()

np.int64(62995)

In [ ]:
df = df.drop(columns=["CustomerID"])

In [ ]:
df.head()

,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn,Issue_Level,Delay_Level,Spend_Level
0,22.0,Female,25.0,14.0,4.0,27.0,Basic,Monthly,598.0,9.0,1.0,Medium Issues,High Delay,High Spend
1,30.0,Female,39.0,14.0,5.0,18.0,Standard,Annual,932.0,17.0,1.0,High Issues,Medium Delay,High Spend
2,41.0,Female,28.0,28.0,7.0,13.0,Standard,Monthly,584.0,20.0,0.0,High Issues,Low Delay,High Spend
3,47.0,Male,27.0,10.0,2.0,29.0,Premium,Annual,757.0,21.0,0.0,Low Issues,High Delay,High Spend
4,65.0,Female,49.0,1.0,10.0,8.0,Basic,Monthly,557.0,6.0,1.0,High Issues,Low Delay,High Spend


##### Observation and Conclusion

1. There are multiple duplicates of customerID's but the attributes are different which suggests dataset creater might have accidently reused them that's why I dropped it.

### Important Features using Logistic Regression 
- I used odds ratios to identify the contribution of feature segments to customer churn.



In [ ]:
df.columns.tolist()

['Age',
 'Gender',
 'Tenure',
 'Usage Frequency',
 'Support Calls',
 'Payment Delay',
 'Subscription Type',
 'Contract Length',
 'Total Spend',
 'Last Interaction',
 'Churn',
 'Issue_Level',
 'Delay_Level',
 'Spend_Level']

In [ ]:
# Features
X = df[
    [
        "Issue_Level",
        "Delay_Level",
        "Spend_Level",
        "Contract Length"
    ]
]

# Target
y = df["Churn"]

# One-Hot Encoding
X_encoded = pd.get_dummies(
    X,
    drop_first=True,
    dtype=int
)

X_encoded.head()

,Issue_Level_Low Issues,Issue_Level_Medium Issues,Delay_Level_Low Delay,Delay_Level_Medium Delay,Spend_Level_Low Spend,Contract Length_Monthly,Contract Length_Quarterly
0,0,1,0,0,0,1,0
1,0,0,0,1,0,0,0
2,0,0,1,0,0,1,0
3,1,0,0,0,0,0,0
4,0,0,1,0,0,1,0


In [ ]:
X_encoded.columns.tolist()

['Issue_Level_Low Issues',
 'Issue_Level_Medium Issues',
 'Delay_Level_Low Delay',
 'Delay_Level_Medium Delay',
 'Spend_Level_Low Spend',
 'Contract Length_Monthly',
 'Contract Length_Quarterly']

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.linear_model import LogisticRegression

model1 = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model1.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lb

In [ ]:
y_pred = model1.predict(X_test)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

Accuracy : 0.8851566675243958
Precision: 0.887634380499364
Recall   : 0.9081088789461488
F1 Score : 0.8977549078349135


In [ ]:

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_encoded, y)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lb

In [ ]:
coef_df = pd.DataFrame({
    "Feature": X_encoded.columns,
    "Coefficient": model.coef_[0]
})

coef_df.sort_values(
    by="Coefficient",
    ascending=False
)

,Feature,Coefficient
4,Spend_Level_Low Spend,2.043516
5,Contract Length_Monthly,1.988643
6,Contract Length_Quarterly,-0.012846
1,Issue_Level_Medium Issues,-2.228073
3,Delay_Level_Medium Delay,-2.540315
2,Delay_Level_Low Delay,-2.765522
0,Issue_Level_Low Issues,-2.847519


In [ ]:

odds_df = pd.DataFrame({
    "Feature": X_encoded.columns,
    "Coefficient": model.coef_[0],
    "Odds_Ratio": np.exp(model.coef_[0])
})

odds_df.sort_values(
    by="Odds_Ratio",
    ascending=False
)

,Feature,Coefficient,Odds_Ratio
4,Spend_Level_Low Spend,2.043516,7.717694
5,Contract Length_Monthly,1.988643,7.305611
6,Contract Length_Quarterly,-0.012846,0.987236
1,Issue_Level_Medium Issues,-2.228073,0.107736
3,Delay_Level_Medium Delay,-2.540315,0.078842
2,Delay_Level_Low Delay,-2.765522,0.062943
0,Issue_Level_Low Issues,-2.847519,0.057988


#### Observations 

1. The low spend customers churn 7.7 times higher than high spend customers.
2. The Customers with the monthly contract length churn 7.3 times higher than customer with annual contract length.
    - Where annual and qurterly customers churn almost equivalently.
3. The High issues customers churn 17.54 times higher than customer with low issues.
4. The customers with High delay churn almost 16 times higher than low delay customers.

#### Conclusion
- So, the high churn segments are
    1. Montly contract
    2. Low spend
    3. High issues 
    4. High delay


### Segment Analysis

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X = pd.get_dummies(df[["Contract Length"]], drop_first=True)
y = (df["Spend_Level"] == "Low Spend").astype(int)

model_contract_spend = LogisticRegression(max_iter=1000)
model_contract_spend.fit(X, y)

pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model_contract_spend.coef_[0],
    "Odds_Ratio": np.exp(model_contract_spend.coef_[0])
}).sort_values("Odds_Ratio", ascending=False)

,Feature,Coefficient,Odds_Ratio
0,Contract Length_Monthly,0.859604,2.362225
1,Contract Length_Quarterly,-0.008225,0.991809


#### Observations
- The monthly Contract length customers are 2.36 times more low spend customers than annual/quarterly customers. 

In [ ]:

X = pd.get_dummies(df[["Contract Length"]], drop_first=True)
y = (df["Spend_Level"] == "High Spend").astype(int)

model_contract_spend = LogisticRegression(max_iter=1000)
model_contract_spend.fit(X, y)

pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model_contract_spend.coef_[0],
    "Odds_Ratio": np.exp(model_contract_spend.coef_[0])
}).sort_values("Odds_Ratio", ascending=False)

,Feature,Coefficient,Odds_Ratio
1,Contract Length_Quarterly,0.008225,1.008259
0,Contract Length_Monthly,-0.859604,0.423330


#### Observations
- The annual/quarterly Contract length customers are 2.38 times more high spend customers than monthly customers. 

In [ ]:
X = pd.get_dummies(
    df[["Spend_Level", "Contract Length","Delay_Level"]],
    drop_first=True
)

y = (df["Issue_Level"] == "High Issues").astype(int)

model_issue = LogisticRegression(max_iter=1000)
model_issue.fit(X, y)

pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model_issue.coef_[0],
    "Odds_Ratio": np.exp(model_issue.coef_[0])
}).sort_values("Odds_Ratio", ascending=False)

,Feature,Coefficient,Odds_Ratio
0,Spend_Level_Low Spend,0.903048,2.467111
1,Contract Length_Monthly,0.777611,2.176268
2,Contract Length_Quarterly,-0.007230,0.992796
4,Delay_Level_Medium Delay,-0.764005,0.465797
3,Delay_Level_Low Delay,-0.891879,0.409885


#### Observations

1. The low Spend Customers face 2.46 times higher issues than high spend customers
2. The montly contract length customers face 2.17 time higher issues than annual/quarterly customers
3. The high delay customers face 2.5 times higher than low delay customers


### Conclusion 
- As we can see the customers with high delay face higher issues may be it due they delay the payment more when they face issues.
- Due to lack of time series data this causality cannot be proven 

In [ ]:

X = pd.get_dummies(df[["Issue_Level"]], drop_first=True)
y = (df["Delay_Level"] == "Medium Delay").astype(int)

model_contract_spend = LogisticRegression(max_iter=1000)
model_contract_spend.fit(X, y)

pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model_contract_spend.coef_[0],
    "Odds_Ratio": np.exp(model_contract_spend.coef_[0])
}).sort_values("Odds_Ratio", ascending=False)

,Feature,Coefficient,Odds_Ratio
0,Issue_Level_Low Issues,0.234068,1.263730
1,Issue_Level_Medium Issues,0.147129,1.158504


### Rankng of Importance of Feature
- I used the the Cramér's V for the association strength of features with churn to rank their importance 
-  Ranking can help decide which features should priortised while preventing the churn


In [ ]:
import scipy.stats as stats

def cramers_v(confusion_matrix):
    chi2 = stats.chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    r, c = confusion_matrix.shape
    return np.sqrt(chi2 / (n * (min(r, c) - 1)))

def validate_relationship(df, feature1, feature2):
    
    table = pd.crosstab(df[feature1], df[feature2])

    chi2, p, dof, expected = stats.chi2_contingency(table)

    cv = cramers_v(table)

    print("="*60)
    print(f"{feature1}  →  {feature2}")
    print("="*60)
    print(table)
    print()
    print(f"Chi-Square Statistic : {chi2:.3f}")
    print(f"P-value              : {p:.6f}")
    print(f"Cramer's V           : {cv:.3f}")
    print()

In [ ]:
validate_relationship(
    df,
    "Contract Length",
    "Spend_Level"
)

Contract Length  →  Spend_Level
Spend_Level      High Spend  Low Spend
Contract Length                       
Annual               147155      51453
Monthly               59856      49378
Quarterly            146504      50860

Chi-Square Statistic : 15282.791
P-value              : 0.000000
Cramer's V           : 0.174



In [ ]:
validate_relationship(
    df,
    "Spend_Level",
    "Issue_Level"
)

Spend_Level  →  Issue_Level
Issue_Level  High Issues  Low Issues  Medium Issues
Spend_Level                                        
High Spend        100321      178819          74375
Low Spend          82905       41811          26975

Chi-Square Statistic : 33647.124
P-value              : 0.000000
Cramer's V           : 0.258



In [ ]:
validate_relationship(
    df,
    "Contract Length",
    "Issue_Level"
)

Contract Length  →  Issue_Level
Issue_Level      High Issues  Low Issues  Medium Issues
Contract Length                                        
Annual                 61652       95887          41069
Monthly                60637       29208          19389
Quarterly              60937       95535          40892

Chi-Square Statistic : 23752.102
P-value              : 0.000000
Cramer's V           : 0.153



In [ ]:
validate_relationship(
    df,
    "Issue_Level",
    "Delay_Level"
)

Issue_Level  →  Delay_Level
Delay_Level    High Delay  Low Delay  Medium Delay
Issue_Level                                       
High Issues         63500      88369         31357
Low Issues          28925     146105         45600
Medium Issues       19284      62514         19552

Chi-Square Statistic : 27919.224
P-value              : 0.000000
Cramer's V           : 0.166



In [ ]:
validate_relationship(
    df,
    "Delay_Level",
    "Churn"
)

Delay_Level  →  Churn
Churn            0.0     1.0
Delay_Level                 
High Delay      6478  105231
Low Delay     167960  129028
Medium Delay   50276   46233

Chi-Square Statistic : 87480.685
P-value              : 0.000000
Cramer's V           : 0.416



In [ ]:
validate_relationship(
    df,
    "Spend_Level",
    "Churn"
)

Spend_Level  →  Churn
Churn           0.0     1.0
Spend_Level                
High Spend   207011  146504
Low Spend     17703  133988

Chi-Square Statistic : 94491.019
P-value              : 0.000000
Cramer's V           : 0.432



In [ ]:
validate_relationship(
    df,
    "Issue_Level",
    "Churn"
)

Issue_Level  →  Churn
Churn             0.0     1.0
Issue_Level                  
High Issues     16876  166350
Low Issues     153926   66704
Medium Issues   53912   47438

Chi-Square Statistic : 152535.693
P-value              : 0.000000
Cramer's V           : 0.549



In [ ]:
validate_relationship(
    df,
    "Contract Length",
    "Churn"
)

Contract Length  →  Churn
Churn               0.0    1.0
Contract Length               
Annual           107067  91541
Monthly           10709  98525
Quarterly        106938  90426

Chi-Square Statistic : 67861.647
P-value              : 0.000000
Cramer's V           : 0.367



#### Observation and conclusion
-  The ranking is as follows
 1. Issue level with Cramér's V of 0.549
 2. Spend level with Cramér's V of 0.432
 3. Delay level with Cramér's V of 0.416
 4. Contract Length with Cramér's V of 0.367

## Decision tree analysis

In [ ]:
# Features
X = df[
    [
        "Issue_Level",
        "Delay_Level",
        "Spend_Level",
        "Contract Length"
    ]
]

# Target
y = df["Churn"]

# One-Hot Encoding
X_encoded = pd.get_dummies(
    X,
    drop_first=False,
    dtype=int
)

X_encoded.head()

,Issue_Level_High Issues,Issue_Level_Low Issues,Issue_Level_Medium Issues,Delay_Level_High Delay,Delay_Level_Low Delay,Delay_Level_Medium Delay,Spend_Level_High Spend,Spend_Level_Low Spend,Contract Length_Annual,Contract Length_Monthly,Contract Length_Quarterly
0,0,0,1,1,0,0,1,0,0,1,0
1,1,0,0,0,0,1,1,0,1,0,0
2,1,0,0,0,1,0,1,0,0,1,0
3,0,1,0,1,0,0,1,0,1,0,0
4,1,0,0,0,1,0,1,0,0,1,0


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Train Decision Tree
dt = DecisionTreeClassifier(
    random_state=42,
    max_depth=4  # Keep it interpretable initially
)

dt.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",4
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current no

In [ ]:
y_pred = dt.predict(X_test)
y_prob = dt.predict_proba(X_test)[:, 1]

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC AUC  :", roc_auc_score(y_test, y_prob))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

Accuracy : 0.8967360107677995
Precision: 0.8866703923859846
Recall   : 0.9332964936986399
F1 Score : 0.9093861812623754
ROC AUC  : 0.916432591296153

Confusion Matrix
[[38251  6692]
 [ 3742 52357]]

Classification Report
              precision    recall  f1-score   support

         0.0       0.91      0.85      0.88     44943
         1.0       0.89      0.93      0.91     56099

    accuracy                           0.90    101042
   macro avg       0.90      0.89      0.89    101042
weighted avg       0.90      0.90      0.90    101042



In [ ]:

importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": dt.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance)

                      Feature  Importance
0     Issue_Level_High Issues    0.451667
6      Spend_Level_High Spend    0.261482
9     Contract Length_Monthly    0.164198
3      Delay_Level_High Delay    0.115532
4       Delay_Level_Low Delay    0.006833
5    Delay_Level_Medium Delay    0.000169
7       Spend_Level_Low Spend    0.000113
2   Issue_Level_Medium Issues    0.000004
8      Contract Length_Annual    0.000001
1      Issue_Level_Low Issues    0.000000
10  Contract Length_Quarterly    0.000000


In [ ]:

from sklearn.tree import _tree

tree = dt.tree_
feature_names = list(X_train.columns)

rules = []

def extract_rules(node, conditions):
    
    # If this is a leaf node
    if tree.feature[node] == _tree.TREE_UNDEFINED:

        samples = tree.n_node_samples[node]

        # Class counts
        proportions = tree.value[node][0]
        samples = tree.n_node_samples[node]

        non_churn = round(proportions[0] * samples)
        churn = round(proportions[1] * samples)

        predicted_class = np.argmax(proportions)

        churn_rate = churn / samples

        rules.append({
            "Rule": " AND ".join(conditions),
            "Samples": samples,
            "Non_Churn": non_churn,
            "Churn": churn,
            "Churn_Rate": round(churn_rate*100,2),
            "Predicted_Class": predicted_class
        })

        return

    feature = feature_names[tree.feature[node]]
    threshold = tree.threshold[node]

    # Left child (<= threshold)
    extract_rules(
        tree.children_left[node],
        conditions + [f"{feature} = No"]
    )

    # Right child (> threshold)
    extract_rules(
        tree.children_right[node],
        conditions + [f"{feature} = Yes"]
    )

extract_rules(0, [])

rules_df = pd.DataFrame(rules)

rules_df = rules_df.sort_values(
    by="Churn_Rate",
    ascending=False
)

rules_df

,Rule,Samples,Non_Churn,Churn,Churn_Rate,Predicted_Class
9,Issue_Level_High Issues = Yes AND Delay_Level_...,17145,361,16784,97.89,1
8,Issue_Level_High Issues = Yes AND Delay_Level_...,33721,1207,32514,96.42,1
11,Issue_Level_High Issues = Yes AND Delay_Level_...,8271,369,7902,95.54,1
10,Issue_Level_High Issues = Yes AND Delay_Level_...,16857,1041,15816,93.82,1
7,Issue_Level_High Issues = No AND Spend_Level_H...,7335,463,6872,93.69,1
3,Issue_Level_High Issues = No AND Spend_Level_H...,7273,460,6813,93.68,1
2,Issue_Level_High Issues = No AND Spend_Level_H...,10728,759,9969,92.93,1
15,Issue_Level_High Issues = Yes AND Delay_Level_...,10739,1449,9290,86.51,1
14,Issue_Level_High Issues = Yes AND Delay_Level_...,21107,2912,18195,86.20,1
13,Issue_Level_High Issues = Yes AND Delay_Level_...,12743,1792,10951,85.94,1


In [ ]:
churn_rules = rules_df[
    rules_df["Predicted_Class"] == 1
]

churn_ruleschurn_rules = churn_rules[
    [
        "Business_Rule",
        "Samples",
        "Churn_Rate"
    ]
]

churn_rules

,Rule,Samples,Non_Churn,Churn,Churn_Rate,Predicted_Class
9,Issue_Level_High Issues = Yes AND Delay_Level_...,17145,361,16784,97.89,1
8,Issue_Level_High Issues = Yes AND Delay_Level_...,33721,1207,32514,96.42,1
11,Issue_Level_High Issues = Yes AND Delay_Level_...,8271,369,7902,95.54,1
10,Issue_Level_High Issues = Yes AND Delay_Level_...,16857,1041,15816,93.82,1
7,Issue_Level_High Issues = No AND Spend_Level_H...,7335,463,6872,93.69,1
3,Issue_Level_High Issues = No AND Spend_Level_H...,7273,460,6813,93.68,1
2,Issue_Level_High Issues = No AND Spend_Level_H...,10728,759,9969,92.93,1
15,Issue_Level_High Issues = Yes AND Delay_Level_...,10739,1449,9290,86.51,1
14,Issue_Level_High Issues = Yes AND Delay_Level_...,21107,2912,18195,86.20,1
13,Issue_Level_High Issues = Yes AND Delay_Level_...,12743,1792,10951,85.94,1


In [ ]:
churn_rules = churn_rules[
    (churn_rules["Samples"] >= 30) &
    (churn_rules["Churn_Rate"] >= 80)
]

pd.set_option('display.max_colwidth', None)

churn_rules

,Rule,Samples,Non_Churn,Churn,Churn_Rate,Predicted_Class
9,Issue_Level_High Issues = Yes AND Delay_Level_Low Delay = No AND Delay_Level_Medium Delay = No AND Contract Length_Monthly = Yes,17145,361,16784,97.89,1
8,Issue_Level_High Issues = Yes AND Delay_Level_Low Delay = No AND Delay_Level_Medium Delay = No AND Contract Length_Monthly = No,33721,1207,32514,96.42,1
11,Issue_Level_High Issues = Yes AND Delay_Level_Low Delay = No AND Delay_Level_Medium Delay = Yes AND Contract Length_Monthly = Yes,8271,369,7902,95.54,1
10,Issue_Level_High Issues = Yes AND Delay_Level_Low Delay = No AND Delay_Level_Medium Delay = Yes AND Contract Length_Monthly = No,16857,1041,15816,93.82,1
7,Issue_Level_High Issues = No AND Spend_Level_High Spend = Yes AND Contract Length_Monthly = Yes AND Delay_Level_High Delay = Yes,7335,463,6872,93.69,1
3,Issue_Level_High Issues = No AND Spend_Level_High Spend = No AND Delay_Level_High Delay = Yes AND Issue_Level_Medium Issues = Yes,7273,460,6813,93.68,1
2,Issue_Level_High Issues = No AND Spend_Level_High Spend = No AND Delay_Level_High Delay = Yes AND Issue_Level_Medium Issues = No,10728,759,9969,92.93,1
15,Issue_Level_High Issues = Yes AND Delay_Level_Low Delay = Yes AND Spend_Level_Low Spend = Yes AND Contract Length_Annual = Yes,10739,1449,9290,86.51,1
14,Issue_Level_High Issues = Yes AND Delay_Level_Low Delay = Yes AND Spend_Level_Low Spend = Yes AND Contract Length_Annual = No,21107,2912,18195,86.20,1
13,Issue_Level_High Issues = Yes AND Delay_Level_Low Delay = Yes AND Spend_Level_Low Spend = No AND Contract Length_Monthly = Yes,12743,1792,10951,85.94,1


In [ ]:
print(tree.value[0])
print(tree.n_node_samples[0])

[[0.44479716 0.55520284]]
404164


In [ ]:
def simplify_rule(rule):

    replacements = {
        # ---------------- ISSUE ----------------
        "Issue_Level_High Issues = Yes": "High Issues",
        "Issue_Level_High Issues = No": "",

        "Issue_Level_Medium Issues = Yes": "Medium Issues",
        "Issue_Level_Medium Issues = No": "",

        "Issue_Level_Low Issues = Yes": "Low Issues",
        "Issue_Level_Low Issues = No": "",

        # ---------------- SPEND ----------------
        "Spend_Level_High Spend = Yes": "High Spend",
        "Spend_Level_High Spend = No": "Low Spend",

        "Spend_Level_Low Spend = Yes": "Low Spend",
        "Spend_Level_Low Spend = No": "High Spend",

        # ---------------- CONTRACT ----------------
        "Contract Length_Monthly = Yes": "Monthly Contract",
        "Contract Length_Monthly = No": "",

        "Contract Length_Annual = Yes": "Annual Contract",
        "Contract Length_Annual = No": "",

        "Contract Length_Quarterly = Yes": "Quarterly Contract",
        "Contract Length_Quarterly = No": "",

        # ---------------- DELAY ----------------
        "Delay_Level_High Delay = Yes": "High Delay",
        "Delay_Level_High Delay = No": "",

        "Delay_Level_Medium Delay = Yes": "Medium Delay",
        "Delay_Level_Medium Delay = No": "",

        "Delay_Level_Low Delay = Yes": "Low Delay",
        "Delay_Level_Low Delay = No": "",
    }

    for old, new in replacements.items():
        rule = rule.replace(old, new)

    # Remove empty ANDs
    parts = [p.strip() for p in rule.split("AND") if p.strip() != ""]

    return " AND ".join(parts)

In [ ]:
churn_rules["Business_Rule"] = churn_rules["Rule"].apply(simplify_rule)

In [ ]:
churn_rules = churn_rules[
    [
        "Business_Rule",
        "Samples",
        "Churn_Rate"
    ]
]

churn_rules

,Business_Rule,Samples,Churn_Rate
9,High Issues AND Monthly Contract,17145,97.89
8,High Issues,33721,96.42
11,High Issues AND Medium Delay AND Monthly Contract,8271,95.54
10,High Issues AND Medium Delay,16857,93.82
7,High Spend AND Monthly Contract AND High Delay,7335,93.69
3,Low Spend AND High Delay AND Medium Issues,7273,93.68
2,Low Spend AND High Delay,10728,92.93
15,High Issues AND Low Delay AND Low Spend AND Annual Contract,10739,86.51
14,High Issues AND Low Delay AND Low Spend,21107,86.20
13,High Issues AND Low Delay AND High Spend AND Monthly Contract,12743,85.94


#### Observation and conclusion 
- The Decision Tree did not introduce entirely new churn drivers. Instead, it independently learned high-risk customer profiles that closely aligned with the findings from exploratory analysis and logistic regression. This consistency across descriptive, statistical, and predictive methods increases confidence that High Issues, Monthly Contracts, Low Spend, and High Payment Delay are robust indicators of churn risk.